# 🤖 Treino da palavra de ativação **"Klaus"** (microWakeWord)

Este caderno cria um pequeno modelo que reconhece a palavra **"Klaus"** — o mesmo ficheiro vai servir no app Android e, mais tarde, na placa ESP32‑S3.

**Como funciona:** uma voz sintética (Piper) diz "Klaus" milhares de vezes, em **vários sotaques e línguas** (português do Brasil e de Portugal, italiano, alemão, espanhol, francês e inglês), com velocidades e tons diferentes. Depois o modelo aprende a distinguir "Klaus" de ruído, música, conversas e palavras parecidas ("claro", "Cláudia", "close"…).

### Antes de começar
1. Menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)** → Guardar.
2. Corra as células **pela ordem**, uma de cada vez (botão ▶ à esquerda de cada célula).
3. Depois da **célula 1**, faça **Ambiente de execução → Reiniciar sessão** e continue na célula 2.
4. O treino completo demora **várias horas** no Colab grátis. Pode deixar o separador aberto e voltar mais tarde.
5. No fim, a última célula descarrega o ficheiro **`klaus_wakeword.zip`**. Envie-o ao Claude.

> Os dados de ruído usados no treino têm licenças variadas: o modelo é para **uso pessoal, não comercial**.

In [ ]:
# 1) Instalação (demora uns minutos). DEPOIS: Ambiente de execução -> Reiniciar sessão.
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!git clone -q https://github.com/kahrendt/microWakeWord
!pip install -q -e ./microWakeWord
# O Colab traz o 'datasets' 4.x, que deixou de devolver o áudio como dicionário: fixa a versão 3.6
!pip install -q datasets==3.6.0

# Gerador de amostras de voz (vozes Piper de várias línguas + gerador inglês com 900 vozes)
!pip install -q piper-tts==1.3.0 webrtcvad
!git clone -q --branch v3.2.0 https://github.com/rhasspy/piper-sample-generator
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
print("✅ Instalado. Agora: Ambiente de execução -> Reiniciar sessão, e continue na célula 2.")

In [ ]:
# 2) Configuração: grafias de "Klaus" por língua e vozes a usar.
#    As grafias diferentes fazem a voz sintética pronunciar de formas diferentes.
import os, sys, subprocess, urllib.parse, random, glob

WAKE_WORD = "Klaus"
PSG = "piper-sample-generator"
VOICES_DIR = "voices"
os.makedirs(VOICES_DIR, exist_ok=True)

# (língua, grafias, vozes "codigo_regiao/nome/qualidade", n.º de amostras)
GRUPOS = [
    ("pt_BR", ["Klaus", "Cláus", "Kláus", "Kláuss", "Klaus!", "Klaus?", "Cláus!"],
     ["pt_BR/cadu/medium", "pt_BR/edresson/low", "pt_BR/faber/medium", "pt_BR/jeff/medium"], 1800),
    ("pt_PT", ["Klaus", "Cláus", "Klaus!", "Klaus?"],
     ["pt_PT/tugão/medium"], 400),
    ("it_IT", ["Klaus", "Claus", "Kla-us", "Klaus!", "Klaus?"],
     ["it_IT/paola/medium", "it_IT/riccardo/x_low"], 700),
    ("de_DE", ["Klaus", "Klaus!", "Klaus?"],
     ["de_DE/thorsten/medium", "de_DE/thorsten_emotional/medium", "de_DE/karlsson/low",
      "de_DE/kerstin/low", "de_DE/eva_k/x_low", "de_DE/ramona/low", "de_DE/pavoque/low"], 800),
    ("es", ["Klaus", "Claus", "Kláus", "Klaus!"],
     ["es_ES/sharvard/medium", "es_ES/davefx/medium", "es_MX/ald/medium", "es_AR/daniela/high"], 500),
    ("fr_FR", ["Klaus", "Claousse", "Klaus!"],
     ["fr_FR/siwis/medium", "fr_FR/tom/medium", "fr_FR/gilles/low"], 400),
    ("en_GB", ["Klaus", "Klouse", "Claus", "Klaus!"],
     ["en_GB/vctk/medium", "en_GB/alba/medium", "en_GB/northern_english_male/medium",
      "en_GB/southern_english_female/low", "en_GB/alan/medium"], 800),
]
# Gerador inglês (LibriTTS, centenas de vozes diferentes)
GERADOR_EN = (["Klaus", "Klouse", "Klowss", "Claus", "Klaus!"], 2000)

# Palavras PARECIDAS que NÃO devem acordar o Klaus (aprende a diferença)
PARECIDAS = {
    "pt_BR": ["claro", "Cláudia", "Cláudio", "calça", "casa", "caos", "Paulo", "cláusula",
              "Clara", "aula", "Laos", "causa", "clã", "louça", "Klara"],
    "it_IT": ["chiaro", "Claudio", "Laura", "causa", "casa", "Paolo"],
    "en":    ["close", "clouds", "class", "cloud", "house", "louse", "cows", "clause", "Klara", "glass"],
}

def baixar_voz(spec):
    loc, nome, q = spec.split("/")
    lang = loc.split("_")[0]
    base = f"https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/{lang}/{loc}/{nome}/{q}/{loc}-{nome}-{q}"
    dest = os.path.join(VOICES_DIR, f"{loc}-{nome}-{q}.onnx")
    if not os.path.exists(dest):
        for ext in (".onnx", ".onnx.json"):
            url = urllib.parse.quote(base + ext, safe=":/")
            r = subprocess.run(["wget", "-q", "-O", dest.replace(".onnx", ext), url])
            if r.returncode != 0:
                print("   ⚠️ não consegui descarregar", spec)
                for f in (dest, dest + ".json"):
                    if os.path.exists(f): os.remove(f)
                return None
    return dest

def gerar(textos, modelos, n, saida, extra=()):
    os.makedirs(saida, exist_ok=True)
    txt = saida.rstrip("/") + "_textos.txt"
    with open(txt, "w", encoding="utf-8") as f:
        f.write("\n".join(textos))
    cmd = [sys.executable, "-m", "piper_sample_generator", txt, "--max-samples", str(n),
           "--output-dir", saida,
           "--length-scales", "0.8", "0.9", "1.0", "1.15", "1.3",
           "--noise-scales", "0.5", "0.667", "0.8", "1.0",
           "--noise-scale-ws", "0.6", "0.8", "1.0", *extra]
    for m in modelos:
        cmd += ["--model", m]
    subprocess.run(cmd, cwd=None, env={**os.environ, "PYTHONPATH": PSG})
    print(f"   {saida}: {len(glob.glob(saida + '/*.wav'))} amostras")

print("Configuração pronta.")

In [ ]:
# 3) Amostras de "Klaus" em inglês com o gerador de centenas de vozes (usa a GPU).
gerar(GERADOR_EN[0], [f"{PSG}/models/en_US-libritts_r-medium.pt"], GERADOR_EN[1],
      "positivas/en_US_gerador", extra=["--batch-size", "100", "--max-speakers", "800"])

In [ ]:
# 4) Amostras de "Klaus" nas outras línguas e sotaques (vozes Piper).
for lingua, grafias, vozes, n in GRUPOS:
    print("▶", lingua)
    modelos = [m for m in (baixar_voz(v) for v in vozes) if m]
    if modelos:
        gerar(grafias, modelos, n, f"positivas/{lingua}")
total = len(glob.glob("positivas/**/*.wav", recursive=True))
print(f"\n✅ Total de amostras de 'Klaus': {total}")

In [ ]:
# 5) Ouça algumas amostras para confirmar que dizem mesmo "Klaus".
from IPython.display import Audio, display
for pasta in sorted(glob.glob("positivas/*/")):
    ficheiros = glob.glob(pasta + "*.wav")
    if ficheiros:
        f = random.choice(ficheiros)
        print(pasta)
        display(Audio(f))

In [ ]:
# 6) Palavras parecidas (exemplos NEGATIVOS): ensinam o modelo a NÃO acordar com "claro", "Cláudia", "close"...
for v in ("pt_BR/faber/medium", "pt_BR/cadu/medium", "it_IT/paola/medium"):
    baixar_voz(v)
gerar(PARECIDAS["pt_BR"], [os.path.join(VOICES_DIR, f) for f in ("pt_BR-faber-medium.onnx", "pt_BR-cadu-medium.onnx")],
      700, "parecidas/pt_BR")
gerar(PARECIDAS["it_IT"], [os.path.join(VOICES_DIR, "it_IT-paola-medium.onnx")], 300, "parecidas/it_IT")
gerar(PARECIDAS["en"], [f"{PSG}/models/en_US-libritts_r-medium.pt"], 700, "parecidas/en",
      extra=["--batch-size", "100", "--max-speakers", "800"])

In [ ]:
# 7) Descarrega ruído, música e ecos de salas para "sujar" as amostras (como na vida real). Demora!
import datasets, scipy, numpy as np
from pathlib import Path
from tqdm import tqdm

if not os.path.exists("./mit_rirs"):
    os.mkdir("./mit_rirs")
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join("./mit_rirs", name), 16000, (row['audio']['array']*32767).astype(np.int16))

if not os.path.exists("audioset"):
    os.mkdir("audioset")
    fname = "bal_train09.tar"
    !wget -q -O audioset/{fname} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}
    !cd audioset && tar -xf bal_train09.tar
    os.makedirs("./audioset_16k", exist_ok=True)
    ds = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join("./audioset_16k", name), 16000, (row['audio']['array']*32767).astype(np.int16))

if not os.path.exists("./fma"):
    os.mkdir("./fma")
    !wget -q -O fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
    !cd fma && unzip -q fma_xs.zip
    os.makedirs("./fma_16k", exist_ok=True)
    ds = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join("./fma_16k", name), 16000, (row['audio']['array']*32767).astype(np.int16))
print("✅ Ruído e ecos prontos")

In [ ]:
# 8) Prepara a "sujidade" (ruído de fundo, eco, volume, tom) aplicada a cada amostra.
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

def clips_de(pasta):
    return Clips(input_directory=pasta, file_pattern='**/*.wav', max_clip_duration_s=None,
                 remove_silence=False, random_split_seed=10, split_count=0.1)

clips_klaus = clips_de("positivas")
clips_parecidas = clips_de("parecidas")

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1, "TanhDistortion": 0.1, "PitchShift": 0.15,
        "BandStopFilter": 0.1, "AddColorNoise": 0.1, "AddBackgroundNoise": 0.75,
        "Gain": 1.0, "RIR": 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5, background_max_snr_db=10,
    min_jitter_s=0.195, max_jitter_s=0.205,
)

# Ouça uma amostra já "suja"
from microwakeword.audio.audio_utils import save_clip
save_clip(augmenter.augment_clip(clips_klaus.get_random_clip()), 'exemplo_sujo.wav')
display(Audio('exemplo_sujo.wav'))

In [ ]:
# 9) Converte as amostras em espectrogramas para o treino (Klaus = positivo, parecidas = negativo).
from mmap_ninja.ragged import RaggedMmap

def gerar_features(clips, out_root, nome):
    for split, split_name, rep, slide in (("training", "train", 2, 10), ("validation", "validation", 1, 10), ("testing", "test", 1, 1)):
        out_dir = os.path.join(out_root, split)
        os.makedirs(out_dir, exist_ok=True)
        if os.path.exists(os.path.join(out_dir, nome)):
            continue
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=slide, step_ms=10)
        RaggedMmap.from_generator(out_dir=os.path.join(out_dir, nome),
                                  sample_generator=sg.spectrogram_generator(split=split_name, repeat=rep),
                                  batch_size=100, verbose=True)

gerar_features(clips_klaus, "features_klaus", "wakeword_mmap")
gerar_features(clips_parecidas, "features_parecidas", "parecidas_mmap")
print("✅ Espectrogramas prontos")

In [ ]:
# 10) Descarrega exemplos negativos prontos (fala, festas, silêncio). Demora!
if not os.path.exists('./negative_datasets'):
    os.mkdir('./negative_datasets')
    root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    for fname in ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']:
        !wget -q -O negative_datasets/{fname} {root}{fname}
        !unzip -q negative_datasets/{fname} -d negative_datasets
print("✅ Negativos prontos")

In [ ]:
# 11) Configuração do treino.
import yaml
config = {
    "window_step_ms": 10,
    "train_dir": "trained_models/klaus",
    "features": [
        {"features_dir": "features_klaus", "sampling_weight": 2.0, "penalty_weight": 1.0,
         "truth": True, "truncation_strategy": "truncate_start", "type": "mmap"},
        {"features_dir": "features_parecidas", "sampling_weight": 4.0, "penalty_weight": 1.5,
         "truth": False, "truncation_strategy": "truncate_start", "type": "mmap"},
        {"features_dir": "negative_datasets/speech", "sampling_weight": 10.0, "penalty_weight": 1.0,
         "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": "negative_datasets/dinner_party", "sampling_weight": 10.0, "penalty_weight": 1.0,
         "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": "negative_datasets/no_speech", "sampling_weight": 5.0, "penalty_weight": 1.0,
         "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": "negative_datasets/dinner_party_eval", "sampling_weight": 0.0, "penalty_weight": 1.0,
         "truth": False, "truncation_strategy": "split", "type": "mmap"},
    ],
    "training_steps": [12000],
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "learning_rates": [0.001],
    "batch_size": 128,
    "time_mask_max_size": [0], "time_mask_count": [0],
    "freq_mask_max_size": [0], "freq_mask_count": [0],
    "eval_step_interval": 500,
    "clip_duration_ms": 1500,
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}
with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f)
print("✅ Configuração gravada")

In [ ]:
# 11b) Correção de memória: limita a validação a 40 000 espectrogramas.
#      Com muitas amostras de "Klaus" a validação passa os 12 GB de RAM do Colab grátis e o treino pára no passo 500.
#      ANTES DA CÉLULA 12: Ambiente de execução -> Reiniciar sessão (liberta a memória das células anteriores).
p = 'microWakeWord/microwakeword/data.py'
s = open(p).read()
a = '        if truncation_strategy != "none":\n            # Spectrograms are all'
b = ('        if mode in ("validation", "testing") and len(data) > 40000:\n'
     '            keep = sorted(random.sample(range(len(data)), 40000))\n'
     '            data = [data[i] for i in keep]\n'
     '            labels = [labels[i] for i in keep]\n'
     '            weights = [weights[i] for i in keep]\n\n')
if 'len(data) > 40000' not in s:
    assert s.count(a) == 1, s.count(a)
    open(p, 'w').write(s.replace(a, b + a))
print('✅ Correção aplicada. Agora: Ambiente de execução -> Reiniciar sessão, e corra a célula 12.')


In [ ]:
# 12) TREINO. Demora várias horas no Colab. Pode parecer parado durante minutos — é normal.
#     Se a sessão cair, volte a correr esta célula: retoma do último ponto guardado.
!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# 13) Empacota o resultado e descarrega "klaus_wakeword.zip". Envie esse ficheiro ao Claude.
import os, glob, json, shutil, zipfile
os.makedirs("resultado", exist_ok=True)
shutil.copy("trained_models/klaus/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite",
            "resultado/klaus.tflite")
manifest = {
    "type": "micro",
    "wake_word": "Klaus",
    "author": "Renan",
    "website": "https://github.com/familiadousseau/klaus",
    "model": "klaus.tflite",
    "trained_languages": ["pt", "it", "de", "es", "fr", "en"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.97,
        "feature_step_size": 10,
        "sliding_window_size": 5,
        "tensor_arena_size": 30000,
        "minimum_esphome_version": "2024.7.0",
    },
}
with open("resultado/klaus.json", "w") as f:
    json.dump(manifest, f, indent=2)
# Guarda também os resultados dos testes (ajudam a afinar a sensibilidade)
for log in glob.glob("trained_models/klaus/**/*.txt", recursive=True) + glob.glob("trained_models/klaus/**/*.json", recursive=True):
    shutil.copy(log, "resultado/" + os.path.basename(log))
with zipfile.ZipFile("klaus_wakeword.zip", "w") as z:
    for f in glob.glob("resultado/*"):
        z.write(f, os.path.basename(f))
from google.colab import files
files.download("klaus_wakeword.zip")